In [2]:
#imports
import snowflake.connector
import pandas as pd
import json
import os

# def df(conn,sql):
#     connCur = conn.cursor()
#     connCur.execute(sql)
#     df = pd.DataFrame(connCur.fetch_pandas_all())
#     connCur.close()
#     return df



c:\Users\214225\AppData\Local\Programs\Python\Python37\lib\site-packages\snowflake\connector\vendored\requests\__init__.py:104: RequestsDependencyWarning: urllib3 (1.26.6) or chardet (5.0.0)/charset_normalizer (2.0.9) doesn't match a supported version!
  RequestsDependencyWarning)


In [4]:
user = 'varun.x.tiwari@us.tel.com'

conn = snowflake.connector.connect(user=user,
                                    account="telus-global.privatelink", 
                                    client_session_keep_alive = False,
                                    warehouse= "developer_wh", 
                                    role= "sysadmin_dev",
                                    threads= 1,    
                                    authenticator="externalbrowser",
                                    autocommit=True)  



C:\Users\221291\AppData\Local\Programs\Python\Python37\lib\site-packages\requests\__init__.py:104: RequestsDependencyWarning: urllib3 (1.26.8) or chardet (5.1.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  RequestsDependencyWarning)


In [31]:
sql = """select top 1 jsonfield from lz_dev.sapapi.ZCDV_T_CMTU084_A350;"""
df_lz = df(conn,sql)

stdf = df_lz['JSONFIELD'].apply(json.loads)
df_lz = pd.DataFrame(stdf.tolist()) 

df_lz.head()




,Datab,Datbi,Kappl,Kbstat,Kfrst,Knumh,Kschl,Vkorg,update_date
0,20220401,99991231,V,,,0041335218,ZKV0,3200,20221104


In [32]:
sql_ods = """select  top 1 * from ods_dev.sapapi.ZCDV_T_CMTU084_A350;"""
df_ods = df(conn,sql_ods)

df_ods.head()

,KAPPL,KSCHL,VKORG,KFRST,DATBI,DATAB,KBSTAT,KNUMH,UPDATE_DATE,_FILEDATE,_FILENAME,AUDIT_LOG_ID,_ROW_HASH
0,V,ZKV0,3200,,99991231,20220401,,0041335218,20221104,2023-01-24 03:20:25,F:SharesFilesSAPH-APIZCDV_T_CMTU084.2023-01-24...,8e9da788-7eb9-4d39-99b3-2794cf7ca8f6,78ff2ff294750ead6900f5dd78bd1668958218d893c5f3...


In [33]:
df_lz.columns = df_lz.columns.str.upper()

lz_ods_diff = df_lz.columns.difference(df_ods.columns)

ods_lz_diff = df_ods.columns.difference(df_lz.columns)

print("LZ_ODS DIFF: {}\nODS_LZ DIFF: {} ".format(lz_ods_diff,ods_lz_diff))


LZ_ODS DIFF: Index([], dtype='object')
ODS_LZ DIFF: Index(['AUDIT_LOG_ID', '_FILEDATE', '_FILENAME', '_ROW_HASH'], dtype='object') 


In [34]:
row_cnt_lz, col_cnt_lz = df_lz.shape
row_cnt_ods, col_cnt_ods = df_ods.shape

lst_ods = []
lst_lz = []
for c in df_ods.columns:
    if (df_ods[c].isnull().sum()) == row_cnt_ods:
        lst_ods.append(c)
print("ODS NULL COLUMN: {}".format(lst_ods))

for c in df_lz.columns:
    if (df_lz[c].isnull().sum()) == row_cnt_lz:
        lst_lz.append(c)
print("LZ NULL COLUMN: {}".format(lst_lz))


ODS NULL COLUMN: []
LZ NULL COLUMN: []


In [5]:
def getFileList(path):
    # get file list for path
    filesList = []
    for x in os.listdir(path):
        z = x.upper()
        if z.endswith('.CSV') and z.startswith('ZCDV'):
            filesList.append(path + "\\" + x)
    return filesList

repo_path = "C:\\Repo\\EnterpriseCloud\\"
templatePath = repo_path + "Notebooks\\CSVTemplates"
refTextPath = repo_path + "Notebooks"

filesList = getFileList(templatePath)

for file in filesList:
    columnList = ''
    csvData = pd.read_csv(file)
    tableName = csvData.at[0,"CDS View"] + "_" + csvData.at[0,"Source.Name"]
    print(tableName)
    template_fields = csvData['Technical Field Name(ODS)'].tolist()


    conn = snowflake.connector.connect(user=user,
                                    account="telus-global.privatelink", 
                                    client_session_keep_alive = False,
                                    warehouse= "developer_wh", 
                                    role= "sysadmin_dev",
                                    threads= 1,    
                                    authenticator="externalbrowser",
                                    autocommit=True)
    
    sql = """select top 1 jsonfield from lz_dev.sapapi."""+tableName+""";"""
    df_lz = df(conn,sql)

    stdf = df_lz['JSONFIELD'].apply(json.loads)
    df_lz = pd.DataFrame(stdf.tolist())
    df_lz.columns = df_lz.columns.str.upper()

    json_fields = df_lz.columns.tolist()
    s = set(template_fields)
    temp3 = [x for x in json_fields if x not in s]
    print(temp3)
    # print("List of json columns {}\n List of template columns {}".format(template_fields,json_fields))      

ZCDV_T_CMTU082_A032
[]
ZCDV_T_CMTU084_A350
['UPDATE_DATE']
ZCDV_T_CMTU086_A919
[]
ZCDV_T_CMTU183_ZSDT0013
['CDATE', 'CPGM', 'CTIME', 'CUSER', 'DELFLG', 'UDATE', 'UPGM', 'UTIME', 'UUSER', 'ZACTODOCDTF', 'ZEXPODOCDTF', 'ZHSUCMPLDT', 'ZHSUCMPLFLG', 'ZMOMAINEMP', 'ZMOSCHEEMP', 'ZPSPNR', 'ZRCVEXPDDTF', 'ZSL1CMPLDT', 'ZSL1EXPDDT', 'ZSL2CMPLDT', 'ZSL2EXPDDT', 'ZSUMAINEMP', 'ZSUSCHEEMP', 'ZTSRREASONF', 'ZTSRSIGNDTF', 'ZTSRSIGNFLGF']
ZCDV_T_CMTU191_FPLA
['DATAAGING', 'DBKEY', 'IFRS15RELEVANCE', 'UPDATE_DATE']


In [186]:
conn.close()

In [75]:
import re
def getFileList(path):
    # get file list for path
    filesList = []
    for x in os.listdir(path):
        z = x.upper()
        if z.endswith('.SQL') and z.startswith('ZCDV'):
            filesList.append(path + "\\" + x)
    return filesList

repo_path = "C:\\Repo\\EnterpriseCloud\\dbt\\lz_to_ods\\models\\sapapi" 

union_sql = ''
filesList = getFileList(repo_path)
table_count = []
for file in filesList:
    tablename = file.replace("C:\Repo\EnterpriseCloud\dbt\lz_to_ods\models\sapapi\\",'').replace(".sql",'')
    base_table_name = re.search('(?<=\\d{3}.).*', tablename).group(0)
    # print(tablename)

    
    sql = """select count(*),'"""+base_table_name+"""' AS tablename,'"""+tablename+"""' AS cdsName  from lz_dev.sapapi."""+tablename+""" UNION\n"""
    union_sql = union_sql + sql
    # df_lz = df(conn,sql)
print(union_sql)



Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
select count(*),'VBAK' AS tablename,'ZCDV_T_CMTU009_VBAK' AS cdsName  from lz_dev.sapapi.ZCDV_T_CMTU009_VBAK UNION
select count(*),'VBAP' AS tablename,'ZCDV_T_CMTU010_VBAP' AS cdsName  from lz_dev.sapapi.ZCDV_T_CMTU010_VBAP UNION
select count(*),'VBEP' AS tablename,'ZCDV_T_CMTU011_VBEP' AS cdsName  from lz_dev.sapapi.ZCDV_T_CMTU011_VBEP UNION
select count(*),'LIKP' AS tablename,'ZCDV_T_CMTU012_LIKP' AS cdsName  from lz_dev.sapapi.ZCDV_T_CMTU012_LIKP UNION
select count(*),'LIPS' AS tablename,'ZCDV_T_CMTU013_LIPS' AS cdsName  from lz_dev.sapapi.ZCDV_T_CMTU013_LIPS UNION
select count(*),'VBRK' AS tablename,'ZCDV_T_CMTU014_VBRK' AS cdsName  from lz_dev.sapapi.ZCDV_T_CMTU014_VBRK UNION
select count(*),'VBRP' AS tablename,'ZCDV_T_CMTU015_VBRP' AS cdsName  from l

In [ ]:
import sys
import fileinput
 
x = ['C_DATE', 'C_PGM', 'C_TIME', 'C_USER', 'DEL_FLG',
       'U_DATE', 'U_PGM', 'U_TIME', 'U_USER', 'Z_ACT_ODOC_DT_F',
       'Z_EXP_ODOC_DT_F', 'Z_HSU_CMPL_DT', 'Z_HSU_CMPL_FLG', 'Z_MO_MAIN_EMP',
       'Z_MO_SCHE_EMP', 'Z_PSPNR', 'Z_RCV_EXPD_DT_F', 'Z_SL1_CMPL_DT',
       'Z_SL1_EXPD_DT', 'Z_SL2_CMPL_DT', 'Z_SL2_EXPD_DT', 'Z_SU_MAIN_EMP',
       'Z_SU_SCHE_EMP', 'Z_TSR_REASON_F', 'Z_TSR_SIGN_DT_F',
       'Z_TSR_SIGN_FLG_F']
y = ['CDATE', 'CPGM', 'CTIME', 'CUSER', 'DELFLG', 'UDATE', 'UPGM', 'UTIME',
       'UUSER', 'ZACTODOCDTF', 'ZEXPODOCDTF', 'ZHSUCMPLDT', 'ZHSUCMPLFLG',
       'ZMOMAINEMP', 'ZMOSCHEEMP', 'ZPSPNR', 'ZRCVEXPDDTF', 'ZSL1CMPLDT',
       'ZSL1EXPDDT', 'ZSL2CMPLDT', 'ZSL2EXPDDT', 'ZSUMAINEMP', 'ZSUSCHEEMP',
       'ZTSRREASONF', 'ZTSRSIGNDTF', 'ZTSRSIGNFLGF']

for l in fileinput.input(files = "C:\\Repo\\EnterpriseCloud\\Notebooks\\CSVTemplates\\zcdv_t_cmtu183-ZSDT0013-full.csv"):
    
    for i in range(len(x)):
        if x[i] in l:
              l = l.replace(x[i], y[i])
    sys.stdout.write(l)

In [76]:
conn.close()

In [15]:
with open('C:\Source\Repos\EnterpriseCloud\Configuration\TelTools.json','r') as f:
        # ora_con=json.load(f)
        # tel_tools_txt = f.read()
        tel_tools_json = json.load(f,strict=False)
        sapapi_json = tel_tools_json.get('Services').get('SAPAPI').get('PROD')
        APP_ID = sapapi_json.get('APP_ID')

        

0b9a4baf
